In [1]:
import numpy as np
import rasterio
from rasterio.transform import from_origin
import matplotlib.pyplot as plt

In [2]:
def generate_valley_lake_dem(Lx=1500, Ly=500, max_depth=70, ext=250, dx=5, dy=5, filename="finallakedem5m.tif"):
    
    """
    Lx: Lake length in x direction (m)
    Ly: Lake width in y direction (m)
    max_depth: Maximum lake depth (m)
    ext: Extra terrain around lake (m); Landslide falls from here into lake
    dx, dy: Grid resolution (m)
    filename: Output TIFF name
    """

    # Change extension to 250 m. Previously it was 100m but due to the deformation of masses near edges, we noticed some irregularities, as in solid volume artificially 
    # increasing!. 500m on the other hand is way too large. 250m just feels perfect, don't ask why

    # Making the grid for DEM
    x = np.arange(-ext, Lx + ext + dx, dx)
    y = np.arange(-ext, Ly + ext + dy, dy)
    X, Y = np.meshgrid(x, y, indexing="ij") # Creates 2D coord matrices
    
    # This is the elevation array. Initially everything is at z = 0
    Z = np.zeros_like(X, dtype=float)

    # Making Lake bathymetry: 40% of total lake is ramp! and 20% of total is the back ramp
    taper = 0.2 * Lx
    Z_long = np.where(X < taper, -max_depth * (X / taper), 
             np.where(X < 0.8 * Lx, -max_depth, -max_depth * (1 - (X - 0.8 * Lx) / (0.2 * Lx))))
    
    # 20% of the lake width is the bank sloping zone on each side
    bank_width = 0.2 * Ly
    Y_scale = np.ones_like(Y)
    Y_scale = np.where(Y < bank_width, Y / bank_width, Y_scale)
    Y_scale = np.where(Y > Ly - bank_width, (Ly - Y) / bank_width, Y_scale)
    
    lake_mask = (X >= 0) & (X <= Lx) & (Y >= 0) & (Y <= Ly)
    Z[lake_mask] = Z_long[lake_mask] * Y_scale[lake_mask]

    # Ramp Extension 
    ramp_slope = max_depth / (0.2 * Lx)
    ramp_ext = (X - Lx) * ramp_slope
    
    ramp_mask = (X > Lx) & (Y >= 0) & (Y <= Ly)
    Z[ramp_mask] = ramp_ext[ramp_mask]

    # Makung the surrounding terrain
    shoreline_Z = np.where(X <= Lx, 0, ramp_ext) 
    
    dist_y_left = np.maximum(0 - Y, 0)
    dist_y_right = np.maximum(Y - Ly, 0)
    dist_y = np.maximum(dist_y_left, dist_y_right)
    
    dist_x_wall = np.maximum(0 - X, 0)
    
    # Banks rising from the Y-sides
    Z_ext_y = shoreline_Z + dist_y * np.tan(np.radians(50))
    
    # Wall rising from the X=0 side
    Z_ext_x = dist_x_wall * np.tan(np.radians(50))
    
    # Apply to everything outside the main lake and ramp channel
    outside_mask = ~lake_mask & ~ramp_mask
    Z[outside_mask] = np.maximum(Z_ext_x, Z_ext_y)[outside_mask]

    Z_export = np.flipud(Z.T)
    transform = from_origin(-ext, Ly + ext, dx, dy)
    
    with rasterio.open(
        filename,
        'w',
        driver='GTiff',
        height=Z_export.shape[0],
        width=Z_export.shape[1],
        count=1,
        dtype=Z_export.dtype,
        crs='EPSG:32643', 
        transform=transform,
    ) as dst:
        dst.write(Z_export, 1)

    print(f"DEM successfully saved to {filename}")
    return x, y, Z

# Execute the generation
x, y, Z = generate_valley_lake_dem()

DEM successfully saved to finallakedem5m.tif


In [3]:
%matplotlib qt

In [4]:
def plot_valley_bathymetry(x, y, Z, water_level=0, stride=5):
    """
    Plots the generated valley and lake bathymetry in 3D using matplotlib.
    stride: Reduces the number of points plotted for better rendering performance.
    """
    X, Y = np.meshgrid(x, y, indexing="ij")
    
    fig = plt.figure(figsize=(10, 7))
    ax = fig.add_subplot(111, projection='3d')
    
    # Plot the terrain/bed
    bed = ax.plot_surface(X[::stride, ::stride], 
                          Y[::stride, ::stride], 
                          Z[::stride, ::stride], 
                          cmap='terrain', edgecolor='none', alpha=0.9)
    
    # Plot the water surface (only where bed elevation is below water level)
    Z_water = np.where(Z < water_level, water_level, np.nan)
    ax.plot_surface(X[::stride, ::stride], 
                    Y[::stride, ::stride], 
                    Z_water[::stride, ::stride], 
                    color='dodgerblue', alpha=0.5, edgecolor='none')
    
    x_range = x.max() - x.min()
    y_range = y.max() - y.min()
    z_range = Z.max() - Z.min()
    
    ax.set_box_aspect((x_range, y_range, z_range))
    
    ax.set_xlabel('Length (X) [m]')
    ax.set_ylabel('Width (Y) [m]')
    ax.set_zlabel('Elevation (Z) [m]')
    ax.set_title(f'Valley & Lake Bathymetry (Water Level = {water_level}m)')
    
    fig.colorbar(bed, ax=ax, shrink=0.5, label="Elevation (m)")
    
    # Set a good initial viewing angle to see the U-shape and ramp
    ax.view_init(elev=35, azim=-120)
    
    plt.tight_layout()
    plt.show()

# To use it with the variables returned from the previous function:
plot_valley_bathymetry(x, y, Z, stride=5)

In [3]:
dx = 5
dy = 5
cell_area = dx * dy  # 25 m^2 per pixel

# Find all the cells that are underwater
water_mask = Z < 0

# Get the exact depth of water in those cells (absolute value)
water_depths = np.abs(Z[water_mask])

# Calculate Surface Area (Total number of water cells * area of one cell)
surface_area_m2 = len(water_depths) * cell_area

# Calculate Total Volume (Sum of all water depths * area of one cell)
total_volume_m3 = np.sum(water_depths) * cell_area

# Calculate Volumetric Mean Depth (H_eff)
H_mean = total_volume_m3 / surface_area_m2

print(f"Surface Area: {surface_area_m2:,.0f} square meters")
print(f"Total Volume: {total_volume_m3:,.0f} cubic meters")
print(f"Volumetric Mean Depth (H_eff): {H_mean:.2f} meters")

Surface Area: 740,025 square meters
Total Volume: 33,600,000 cubic meters
Volumetric Mean Depth (H_eff): 45.40 meters
